<h1>Deep Explanation: CPNR + DWT</h1>

<h2>Stage 1 — Numerical Encoding (CPNR)</h2>
A DNA sequence is a string of letters. Signal processing requires numbers. So the first question is: how do you map 20 amino acids (or 4 nucleotides) to numbers in a way that preserves biological meaning?

EIIP assigns a fixed physical constant (electron-ion interaction potential) to each amino acid. The problem is degeneracy: two chemically different amino acids can get the same number.

CPNR solves this by mapping each amino acid to a unique prime number, then projecting it onto the unit circle in the complex plane:

x_j = e^(2π·i·f(a_j) / 67)

Where f(a_j) is the prime number for amino acid a_j, and 67 is chosen because it is just above the largest prime used (67 itself for lysine K), keeping all 20 points well-separated on the unit circle. The result is a complex-valued sequence where each point has magnitude 1 but a unique angle — no two amino acids overlap.

For DNA we work at the nucleotide level directly (4 bases: A, C, G, T) using the EIIP values, and at the codon/amino acid level using CPNR.

<h2>Stage 2 — From Sequence to Signal</h2>
After encoding, your protein/DNA sequence becomes a discrete-time complex signal x[n]. For a window of 2100 nucleotides this is an array of 2100 complex numbers. This is your "genomic signal."

<h2>Stage 3 — DFT and the Resonant Recognition Model</h2>
The Resonant Recognition Model (RRM) says that proteins with the same biological function share a characteristic frequency in the power spectrum of their EIIP/CPNR encoding. You compute the DFT to find this:

X[k] = Σ x[n] · e^(-2πi·kn/N)

The cross-spectrum between a query sequence and a reference set reveals the characteristic frequency as a peak. Amino acids near that peak location are the hotspots.

<h2>Stage 4 — Why DWT Instead of (or in Addition to) DFT</h2>
DFT gives you frequency content but loses all positional information — you know a frequency exists in the signal but not where. For a 30,000-nucleotide coronavirus genome, knowing that frequency 0.3 is present tells you nothing about which of the 10,000 codons it comes from.
DWT solves this through multi-resolution analysis. Think of it like a filter bank:


Original signal (2100 samples)
       │
   [HPF]──── Detail coefficients D1 (high freq, 1050 samples)
       │
   [LPF]──── Approximation A1 (low freq, 1050 samples)
                    │
                [HPF]──── Detail D2 (525 samples)
                    │
                [LPF]──── Approximation A2 (525 samples)
                                │
                            [HPF]──── Detail D3
                                │
                            [LPF]──── Approximation A3

At each level the signal is split into a high-frequency detail (HPF) and a low-frequency approximation (LPF), each subsampled by 2. The key insight is that each coefficient knows both its frequency band AND its approximate position in the original signal. High-level details (D1) have fine time resolution but coarse frequency resolution. Low-level approximations (A3, A4) have coarse time resolution but fine frequency resolution.

For genomic hotspot detection: hotspot regions create energy concentrations in specific DWT subbands. The peak energies across subbands form your feature vector.

<h2>Stage 5 — Feature Vector Construction</h2>
From a window of length N, after CPNR encoding and DWT decomposition to level L, you extract:

- Energy of each detail subband: E_Di = Σ |Di[k]|²
- Energy of final approximation: E_AL
- Statistical moments of each subband: mean, std, skewness, kurtosis
- Peak location in each subband (hotspot position)

This gives you a fixed-length feature vector regardless of the original sequence length — which is exactly what we need for the classifier.

In [1]:
"""
cpnr_eiip.py

Feature Stream A: Numerical encoding + DWT feature extraction.

Pipeline per sequence window:
  1. Encode nucleotides/codons using EIIP or CPNR
  2. Compute DWT decomposition
  3. Extract energy + statistical features from each subband

Deep explanation of every design choice is in the docstrings.
"""

import numpy as np
import pywt
from scipy.stats import skew, kurtosis
from typing import Literal

import pandas as pd


# ════════════════════════════════════════════════════════════════
# PART 1 — ENCODING TABLES
# ════════════════════════════════════════════════════════════════

# ── EIIP values for 20 amino acids ──────────────────────────────
# Source: Cosic (1994), IEEE Trans. Biomed. Eng.
# These are physical constants: the electron-ion interaction
# pseudopotential of each amino acid, measured in Rydbergs.
# They represent the free electron energy of each residue,
# which correlates with its electromagnetic resonance.
EIIP_AMINO_ACID = {
    "M": 0.0823, "W": 0.0548, "F": 0.0946, "Y": 0.0516,
    "P": 0.0198, "C": 0.0829, "T": 0.0941, "H": 0.0242,
    "V": 0.0057, "L": 0.0000, "Q": 0.0761, "S": 0.0829,
    "A": 0.0373, "N": 0.0036, "G": 0.0050, "R": 0.0959,
    "I": 0.0000, "D": 0.1263, "E": 0.0058, "K": 0.0371,
}

# ── EIIP values for 4 DNA nucleotides ───────────────────────────
# Used when processing DNA directly (not translated to protein)
# Source: Nair & Sreenivasan (2006)
EIIP_NUCLEOTIDE = {
    "A": 0.1260, "T": 0.1335, "G": 0.0806, "C": 0.1340,
}

# ── CPNR prime numbers for 20 amino acids ───────────────────────
# Source: Chen et al., J. Comput. Biol. (2016)
#
# Design principle: each amino acid is assigned a prime number
# such that amino acids with the same number of codons share
# primes with the same gap to the next prime (see Table 16.5
# in Paper 1). This encodes codon degeneracy information.
#
# The 20 primes are selected from {1,2,3,5,7,11,...,67}.
# They are then projected onto the unit circle:
#   x_j = exp(2π·i·f(a_j) / 67)
# so all amplitudes equal 1 and only angles differ.
CPNR_PRIME = {
    "M":  1,  "W":  2,  "F":  3,  "Y":  5,
    "P":  7,  "C": 11,  "T": 13,  "H": 17,
    "V": 19,  "L": 23,  "Q": 29,  "S": 31,
    "A": 37,  "N": 41,  "G": 43,  "R": 47,
    "I": 53,  "D": 59,  "E": 61,  "K": 67,
}
CPNR_MODULUS = 67  # denominator in the unit-circle formula

# ── Standard genetic code: codon → amino acid ───────────────────
# Maps every 3-nucleotide codon to its single-letter amino acid.
# Stop codons are mapped to None.
CODON_TABLE = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "TAT": "Y", "TAC": "Y", "TAA": None, "TAG": None,
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "TGT": "C", "TGC": "C", "TGA": None, "TGG": "W",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}


# ════════════════════════════════════════════════════════════════
# PART 2 — ENCODING FUNCTIONS
# ════════════════════════════════════════════════════════════════

def encode_eiip_nucleotide(sequence: str) -> np.ndarray:
    """
    Encodes a DNA sequence using EIIP nucleotide values.

    Returns a real-valued 1D array of length N (one value per nt).

    This is the simplest encoding: each nucleotide gets its
    pseudopotential value. The resulting signal is real-valued
    and can be fed directly into DWT.

    Example:
      "ATCG" → [0.1260, 0.1335, 0.1340, 0.0806]
    """
    return np.array(
        [EIIP_NUCLEOTIDE.get(c, 0.0) for c in sequence.upper()],
        dtype=np.float64,
    )


def encode_eiip_codon(sequence: str) -> np.ndarray:
    """
    Encodes a DNA sequence by translating codons to amino acids
    and then applying EIIP amino acid values.

    Returns a real-valued 1D array of length N//3
    (one value per codon).

    The sequence length must be divisible by 3 (codon-aligned).
    Stop codons are assigned value 0.0.

    Why codon-level?
      The biological information in DNA is carried in triplets.
      Processing at the codon level reduces signal length by 3×
      while preserving the functional units. This is the level
      used for hotspot detection in Paper 1.
    """
    seq = sequence.upper()
    n_codons = len(seq) // 3
    values = np.zeros(n_codons, dtype=np.float64)

    for i in range(n_codons):
        codon = seq[i*3 : i*3 + 3]
        aa = CODON_TABLE.get(codon, None)
        if aa is not None:
            values[i] = EIIP_AMINO_ACID.get(aa, 0.0)

    return values


def encode_cpnr(sequence: str) -> np.ndarray:
    """
    Encodes a DNA sequence at the codon level using CPNR.

    Returns a COMPLEX-valued 1D array of length N//3.
    Each element lies on the unit circle: |x_j| = 1.

    Formula:
      x_j = exp(2π·i·f(a_j) / 67)

    where f(a_j) is the prime number for the amino acid
    encoded by the j-th codon.

    Why complex?
      Real-valued encodings (EIIP) suffer from the degeneracy
      problem: two amino acids can get the same value, making
      them indistinguishable in the signal. By projecting to the
      unit circle with unique prime-number angles, each amino
      acid occupies a unique point in the complex plane —
      guaranteed to be distinct because prime numbers ensure
      no two fractions f(a)/67 are equal.

    Why unit circle?
      It normalises amplitude differences between amino acids.
      With raw prime numbers (1–67), lysine (K=67) would
      dominate the signal simply due to its large value, not
      biological importance. On the unit circle, every amino
      acid has equal amplitude and only angle differs.

    Stop codons are encoded as (1+0j) — the real unit, which
    is a neutral position on the circle far from any amino acid.
    """
    seq = sequence.upper()
    n_codons = len(seq) // 3
    values = np.zeros(n_codons, dtype=np.complex128)

    for i in range(n_codons):
        codon = seq[i*3 : i*3 + 3]
        aa = CODON_TABLE.get(codon, None)
        if aa is not None:
            prime = CPNR_PRIME.get(aa, 1)
            # Unit circle projection: e^(2πi·p/67)
            angle = 2 * np.pi * prime / CPNR_MODULUS
            values[i] = np.exp(1j * angle)
        else:
            values[i] = 1.0 + 0j   # stop codon: neutral

    return values


def encode_sequence(
    sequence: str,
    method: Literal["eiip_nt", "eiip_codon", "cpnr"] = "cpnr",
) -> np.ndarray:
    """
    Unified encoding dispatcher.

    Parameters
    ----------
    sequence : cleaned nucleotide string (must be ACGT only)
    method   : encoding to use:
               "eiip_nt"    — EIIP per nucleotide (real, length N)
               "eiip_codon" — EIIP per codon (real, length N//3)
               "cpnr"       — CPNR per codon (complex, length N//3)

    Returns
    -------
    numpy array (real for eiip_*, complex for cpnr)
    """
    if method == "eiip_nt":
        return encode_eiip_nucleotide(sequence)
    elif method == "eiip_codon":
        return encode_eiip_codon(sequence)
    elif method == "cpnr":
        return encode_cpnr(sequence)
    else:
        raise ValueError(
            f"Unknown method '{method}'. "
            "Choose from: eiip_nt, eiip_codon, cpnr"
        )


# ════════════════════════════════════════════════════════════════
# PART 3 — DWT DECOMPOSITION
# ════════════════════════════════════════════════════════════════

# Recommended wavelets for genomic signals
# (from literature, Paper 1 uses bior3.9)
RECOMMENDED_WAVELETS = {
    "bior3.9" : "Biorthogonal 3.9 — best for hotspot energy (Paper 1)",
    "db4"     : "Daubechies 4 — smooth, good for codon patterns",
    "sym5"    : "Symlet 5 — near-symmetric, good for protein signals",
    "haar"    : "Haar — simplest, fast, discontinuity detection",
    "coif3"   : "Coiflet 3 — good for oscillatory biological signals",
}


def dwt_decompose(
    signal: np.ndarray,
    wavelet: str = "bior3.9",
    level: int = None,
    mode: str = "periodization",
) -> tuple[list, list]:
    """
    Performs multilevel Discrete Wavelet Transform decomposition.

    For complex CPNR signals, decomposes real and imaginary
    parts separately and recombines — this is mathematically
    equivalent to complex DWT but uses the real pywt library.

    Parameters
    ----------
    signal  : encoded sequence (real or complex numpy array)
    wavelet : mother wavelet name (see RECOMMENDED_WAVELETS)
    level   : decomposition depth. If None, uses maximum
              level for this signal length and wavelet.
              We cap at 6 to avoid over-decomposition.
    mode    : boundary extension mode:
              "periodization" — signal treated as periodic,
              gives fewest coefficients (no boundary artefacts
              in length), best for fixed-window features.

    Returns
    -------
    (coeffs_real, coeffs_imag) where each is a list:
        [cA_L, cD_L, cD_{L-1}, ..., cD_1]
    For real signals, coeffs_imag is a list of zero arrays.

    Coefficient structure (level=3 example):
        coeffs[0] = cA3  (approximation at level 3, lowest freq)
        coeffs[1] = cD3  (detail at level 3)
        coeffs[2] = cD2  (detail at level 2)
        coeffs[3] = cD1  (detail at level 1, highest freq)

    Frequency interpretation (for signal sampled at rate Fs):
        cD1: Fs/4  to Fs/2   (highest frequency detail)
        cD2: Fs/8  to Fs/4
        cD3: Fs/16 to Fs/8
        cA3: 0     to Fs/16  (lowest frequency approximation)
    """
    wavelet_obj = pywt.Wavelet(wavelet)

    # Determine decomposition level
    max_level = pywt.dwt_max_level(len(signal), wavelet_obj)
    if level is None:
        level = min(max_level, 6)
    else:
        level = min(level, max_level)

    if np.iscomplexobj(signal):
        # Decompose real and imaginary parts separately
        coeffs_real = pywt.wavedec(
            signal.real, wavelet_obj, mode=mode, level=level
        )
        coeffs_imag = pywt.wavedec(
            signal.imag, wavelet_obj, mode=mode, level=level
        )
    else:
        coeffs_real = pywt.wavedec(
            signal.real, wavelet_obj, mode=mode, level=level
        )
        coeffs_imag = [np.zeros_like(c) for c in coeffs_real]

    return coeffs_real, coeffs_imag


# ════════════════════════════════════════════════════════════════
# PART 4 — FEATURE EXTRACTION FROM DWT SUBBANDS
# ════════════════════════════════════════════════════════════════

def subband_features(coeffs: np.ndarray) -> np.ndarray:
    """
    Extracts a fixed set of statistical features from one
    DWT subband coefficient array.

    Features extracted (6 per subband):
      1. Energy       — Σ|c_i|² / N
                        Captures how much signal power is in
                        this frequency band. Hotspots create
                        localised energy concentrations.

      2. Mean absolute value — Σ|c_i| / N
                        Robust to sign changes in coefficients.

      3. Standard deviation   — std of |c_i|
                        Measures variability in the subband.
                        High std = non-uniform energy distribution
                        (potential hotspot regions).

      4. Skewness     — asymmetry of the coefficient distribution.
                        Negative skew = more large-magnitude
                        coefficients (concentrated hotspots).

      5. Kurtosis     — "peakedness" of the distribution.
                        High kurtosis = sparse, localised energy
                        (classic signature of functional sites).

      6. Max absolute value — max(|c_i|)
                        Captures the single strongest response
                        in this frequency band.

    Returns
    -------
    numpy array of shape (6,)
    """
    abs_c = np.abs(coeffs)
    n = len(abs_c)

    if n == 0:
        return np.zeros(6)

    energy   = np.sum(abs_c ** 2) / n
    mean_abs = np.mean(abs_c)
    std_abs  = np.std(abs_c)
    skew_val = float(skew(abs_c))   if n > 2 else 0.0
    kurt_val = float(kurtosis(abs_c)) if n > 2 else 0.0
    max_abs  = np.max(abs_c)

    return np.array(
        [energy, mean_abs, std_abs, skew_val, kurt_val, max_abs],
        dtype=np.float64,
    )


def extract_dwt_features(
    signal: np.ndarray,
    wavelet: str = "bior3.9",
    level: int = 4,
    use_imaginary: bool = True,
) -> np.ndarray:
    """
    Full DWT feature extraction pipeline for one encoded signal.

    For each decomposition level (approximation + details),
    extracts subband_features() from both real and imaginary
    components (if complex signal).

    Feature vector structure (level=4, complex signal):
        - 5 subbands (cA4, cD4, cD3, cD2, cD1)
        - 2 components per subband (real, imag)
        - 6 statistics per component
        Total: 5 × 2 × 6 = 60 features

    For real signal (eiip_*):
        Total: 5 × 1 × 6 = 30 features

    Parameters
    ----------
    signal        : encoded genomic signal (real or complex)
    wavelet       : mother wavelet
    level         : DWT decomposition depth
    use_imaginary : if False, ignores imaginary component
                    even for complex signals

    Returns
    -------
    1D numpy feature vector
    """
    coeffs_real, coeffs_imag = dwt_decompose(
        signal, wavelet=wavelet, level=level
    )

    features = []

    for i, (cr, ci) in enumerate(zip(coeffs_real, coeffs_imag)):
        # Real component features
        features.append(subband_features(cr))

        # Imaginary component (only for CPNR complex signals)
        if use_imaginary and np.any(ci != 0):
            features.append(subband_features(ci))

    return np.concatenate(features)


# ════════════════════════════════════════════════════════════════
# PART 5 — FULL PIPELINE: SEQUENCE → FEATURE VECTOR
# ════════════════════════════════════════════════════════════════

def extract_stream_a_features(
    sequence: str,
    encoding: Literal["eiip_nt", "eiip_codon", "cpnr"] = "cpnr",
    wavelet: str = "bior3.9",
    dwt_level: int = 4,
) -> np.ndarray:
    """
    Complete Feature Stream A pipeline for a single sequence window.

    Steps:
      1. Encode sequence → numerical signal
      2. DWT decompose → subband coefficients
      3. Extract statistical features from each subband
      4. Return fixed-length feature vector

    Parameters
    ----------
    sequence : cleaned, fixed-length nucleotide window (ACGT only)
    encoding : encoding method (see encode_sequence)
    wavelet  : DWT mother wavelet
    dwt_level: decomposition depth

    Returns
    -------
    1D numpy array (feature vector, fixed length)
    """
    # Step 1: Encode
    signal = encode_sequence(sequence, method=encoding)

    # Step 2 + 3: DWT + feature extraction
    features = extract_dwt_features(
        signal,
        wavelet=wavelet,
        level=dwt_level,
        use_imaginary=(encoding == "cpnr"),
    )

    return features


def get_feature_names(
    encoding: str = "cpnr",
    dwt_level: int = 4,
) -> list[str]:
    """
    Returns human-readable names for each feature dimension.
    Useful for feature importance analysis.
    """
    stat_names = [
        "energy", "mean_abs", "std", "skewness",
        "kurtosis", "max_abs"
    ]
    n_subbands = dwt_level + 1   # L details + 1 approximation
    components = (
        ["real", "imag"] if encoding == "cpnr" else ["real"]
    )

    names = []
    subband_labels = (
        [f"cA{dwt_level}"]
        + [f"cD{i}" for i in range(dwt_level, 0, -1)]
    )

    for sub in subband_labels:
        for comp in components:
            for stat in stat_names:
                names.append(f"{sub}_{comp}_{stat}")

    return names


# ════════════════════════════════════════════════════════════════
# PART 6 — BATCH PROCESSING
# ════════════════════════════════════════════════════════════════

def extract_features_dataframe(
    df_windows: pd.DataFrame,
    encoding: str = "cpnr",
    wavelet: str = "bior3.9",
    dwt_level: int = 4,
    verbose: bool = True,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Extracts Stream A features for all windows in a DataFrame.

    Parameters
    ----------
    df_windows : windowed DataFrame from preprocessing
    encoding   : encoding method
    wavelet    : DWT wavelet
    dwt_level  : DWT depth
    verbose    : print progress

    Returns
    -------
    X          : feature matrix, shape (n_windows, n_features)
    y          : label array, shape (n_windows,)
    feat_names : list of feature name strings
    """
    import pandas as pd
    from tqdm import tqdm

    if verbose:
        print(f"\nExtracting Stream A features ...")
        print(f"  Encoding : {encoding}")
        print(f"  Wavelet  : {wavelet}")
        print(f"  DWT level: {dwt_level}")

    sequences = df_windows["sequence"].tolist()
    labels    = df_windows["label_enc"].values

    # Extract features for first sample to get feature dimension
    first_feats = extract_stream_a_features(
        sequences[0], encoding, wavelet, dwt_level
    )
    n_features = len(first_feats)

    if verbose:
        print(f"  Feature vector size: {n_features}")

    X = np.zeros((len(sequences), n_features), dtype=np.float64)
    X[0] = first_feats

    iterator = tqdm(sequences[1:], desc="Feature extraction") \
               if verbose else sequences[1:]

    for i, seq in enumerate(iterator, start=1):
        X[i] = extract_stream_a_features(
            seq, encoding, wavelet, dwt_level
        )

    feat_names = get_feature_names(encoding, dwt_level)

    if verbose:
        print(f"  Done. X shape: {X.shape}")

    return X, labels, feat_names



In [ ]:
df = pd.read_csv("C:\\Users\\rodri\\Desktop\\MASTER\\EMMBIOME\\Grecia\\Courses\\Bioinformatics\\Project\\genomic_multimodal_fusion\\data\\processed\\all_viruses_sequences.csv")


In [2]:

# ── Quick test ───────────────────────────────────────────────────
if __name__ == "__main__":
    import sys
    sys.path.insert(0, ".")
    from src.data_loader import load_virus_dataset
    from src.preprocessing import run_preprocessing_pipeline

    # load and preprocess
    df_raw = load_virus_dataset(
        virus_names=["coronavirus", "influenza"],
        max_per_virus=20,
    )
    results = run_preprocessing_pipeline(df_raw, save=False)
    df_train = results["df_train_windows"]

    # test all three encodings on one sequence
    seq = df_train["sequence"].iloc[0]
    print(f"sequence length : {len(seq)}")
    print(f"virus           : {df_train['virus'].iloc[0]}")

    for enc in ["eiip_nt", "eiip_codon", "cpnr"]:
        signal = encode_sequence(seq, method=enc)
        feats  = extract_stream_a_features(seq, encoding=enc)
        print(f"\n{enc}:")
        print(f"  Signal  : shape={signal.shape}, "
              f"dtype={signal.dtype}")
        print(f"  Features: shape={feats.shape}")
        print(f"  First 5 : {feats[:5].round(6)}")

    # Full batch extraction
    print("\nBatch extraction on train set:")
    X, y, names = extract_features_dataframe(df_train)
    print(f"  X: {X.shape}, y: {y.shape}")
    print(f"  Feature names (first 5): {names[:5]}")
    print(f"  Any NaN: {np.isnan(X).any()}")
    print(f"  Any Inf: {np.isinf(X).any()}")

ModuleNotFoundError: No module named 'src'